In [1]:
# ============================================
# 1. Install required libraries
# ============================================

!pip install -q langchain-openai
!pip install -q langchain-tavily tavily-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.0/345.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
# ============================================
# 2. Imports
# ============================================

import os
import requests
import smtplib

from email.mime.text import MIMEText
from IPython.display import display, Markdown
from langchain_openai import ChatOpenAI
from kaggle_secrets import UserSecretsClient
from langchain_tavily import TavilySearch
from langchain.tools import tool
from langchain.agents import create_agent


# ============================================
# 3. Get Secrets from Kaggle
# ============================================

user_secrets = UserSecretsClient()

OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")
TAVILY_API_KEY = user_secrets.get_secret("TAVILY_API_KEY")
TELEGRAM_BOT_TOKEN = user_secrets.get_secret("TELEGRAM_BOT_TOKEN")
TELEGRAM_USER_ID = user_secrets.get_secret("TELEGRAM_USER_ID")
GMAIL_APP_PASSWORD = user_secrets.get_secret("GMAIL_APP_PASSWORD")


# ============================================
# 4. Configure APIs
# ============================================

os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
# ============================================
# 5. Configure LLM
# ============================================

llm = ChatOpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free"
)

In [4]:
# ============================================
# 6. Tools
# ============================================

@tool
def send_telegram(msg: str) -> str:
    """Send a message to the user's Telegram."""

    url = (
        f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}"
        f"/sendMessage"
    )

    response = requests.post(
        url,
        data={
            "chat_id": TELEGRAM_USER_ID,
            "text": msg
        }
    )

    if response.ok:
        return "Telegram message sent successfully."

    return f"Telegram failed: {response.text}"


@tool
def send_email(
    receiver_email: str,
    subject: str,
    message: str
) -> str:
    """Send an email using Gmail with UTF-8 support."""

    sender_email = "zidanmalak999@gmail.com"

    msg = MIMEText(
        message,
        "plain",
        "utf-8"
    )

    msg["Subject"] = subject
    msg["From"] = sender_email
    msg["To"] = receiver_email

    try:
        server = smtplib.SMTP(
            "smtp.gmail.com",
            587
        )

        server.starttls()

        server.login(
            sender_email,
            GMAIL_APP_PASSWORD
        )

        server.sendmail(
            sender_email,
            receiver_email,
            msg.as_string()
        )

        server.quit()

        return "Email sent successfully."

    except Exception as e:
        return f"Email failed: {str(e)}"


@tool
def web_search(query: str) -> str:
    """Search the web for recent AI news from the last 24 hours."""

    search = TavilySearch(
        max_results=5,
        topic="news",
        search_depth="advanced",
        days=1
    )

    results = search.invoke(
        f"Latest AI news from the last 24 hours: {query}"
    )

    return str(results)


# ============================================
# 7. Tools list
# ============================================

tools = [
    web_search,
    send_email,
    send_telegram
]

In [5]:
# ============================================
# 8. Create Researcher Agent
# ============================================

Researcher_agent = create_agent(
    model=llm,
    tools=[web_search],

    system_prompt="""
You are an AI News Researcher.

Your job is to:
1. Search for the latest important AI news from the last 24 hours.
2. Select the 5 most important and reliable stories.
3. Summarize them accurately.
4. Prepare a professional report for Telegram and Email.

For each news story include:

- A clear headline
- What happened
- Why it matters
- Source

Do not invent information.

========================================
TELEGRAM FORMAT
========================================

Create a concise Arabic Telegram report.

Use this structure:

🤖 AI NEWS DAILY
📅 [DATE]

━━━━━━━━━━━━━━━━━━

🔥 أهم أخبار الذكاء الاصطناعي

1️⃣ [عنوان الخبر]

📰 ماذا حدث؟
[شرح مختصر وواضح]

💡 لماذا يهم؟
[شرح مختصر]

🔗 المصدر: [SOURCE]

━━━━━━━━━━━━━━━━━━

2️⃣ [عنوان الخبر]

📰 ماذا حدث؟
[شرح مختصر]

💡 لماذا يهم؟
[شرح مختصر]

🔗 المصدر: [SOURCE]

...

━━━━━━━━━━━━━━━━━━

📌 الخلاصة
[2-3 important takeaways]

Rules for Arabic:
- Use fluent Modern Standard Arabic.
- Do not translate word-by-word.
- Do not mix Arabic with random foreign languages.
- Company names, product names and technical terms may remain in English.
- Keep the Arabic natural and professional.

========================================
EMAIL FORMAT
========================================

Create a professional English email report.

Use this structure:

🤖 AI NEWS DAILY
📅 [DATE]

━━━━━━━━━━━━━━━━━━━━━━━━

🔥 TOP AI NEWS

1. [Headline]

📰 What happened?
[Explanation]

💡 Why it matters:
[Explanation]

🔗 Source: [SOURCE]

━━━━━━━━━━━━━━━━━━━━━━━━

2. [Headline]

📰 What happened?
[Explanation]

💡 Why it matters:
[Explanation]

🔗 Source: [SOURCE]

...

━━━━━━━━━━━━━━━━━━━━━━━━

📌 KEY TAKEAWAYS

• [Takeaway]
• [Takeaway]
• [Takeaway]

Keep the email concise, professional and easy to scan.

========================================

IMPORTANT:

Return TWO clearly separated sections:

=== TELEGRAM ===
[Arabic Telegram report]

=== EMAIL ===
[English Email report]
"""
)


# ============================================
# 9. Create Sender Agent
# ============================================

Sender_agent = create_agent(
    model=llm,
    tools=[send_email, send_telegram],

    system_prompt="""
You are an AI News Distribution Agent.

You receive a report containing two sections:

=== TELEGRAM ===
Arabic report

=== EMAIL ===
English report

Your job is only to distribute them.

1. Send ONLY the Arabic Telegram section to Telegram.
2. Send ONLY the English Email section to email.
3. Do not rewrite the report.
4. Do not translate anything.
5. Do not add new information.

For the email:
Subject:
🤖 AI News Daily - Latest 24 Hours

For Telegram:
Send the Arabic report exactly as provided.
"""
)

In [6]:
# ============================================
# 10. Run Researcher Agent
# ============================================

research_result = Researcher_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
Research the latest important AI news from the last 24 hours.

Select the 5 most important and reliable stories.

For each story:
- Explain what happened.
- Mention the source when available.
- Explain why it matters.

Then create:
1. A professional English summary for email.
2. A fluent Modern Standard Arabic summary for Telegram.

Return both summaries clearly.
"""
            }
        ]
    }
)


# ============================================
# 11. Extract Research Report
# ============================================

news_report = research_result["messages"][-1].content

print("===== RESEARCH REPORT =====")
display(Markdown(news_report))





===== RESEARCH REPORT =====



=== EMAIL ===

🤖 AI NEWS DAILY
📅 September 21, 2026

━━━━━━━━━━━━━━━━━━━━━━━━

🔥 TOP AI NEWS

1. Anthropic Weighs New Model Counter to OpenAI's GPT-6 Astra

📰 What happened?
Anthropic is reportedly considering accelerating the launch of a new AI model to counter OpenAI's recently released GPT-6 Astra, which has gained traction among enterprise users. This comes despite CEO Dario Amodei's recent public call for an industry-wide slowdown in AI development due to safety concerns. The move highlights mounting competitive pressure as OpenAI's Astra captured roughly 13% of enterprise AI spending tracked by Ramp, compared to 8% for Anthropic's Claude Fable. The potential release would also align with Anthropic's anticipated $2 trillion IPO, possibly delayed to November 2026.

💡 Why it matters:
Demonstrates the tension between AI safety advocacy and commercial competition among frontier labs. OpenAI's latest model appears to be gaining market share, pressuring rivals to respond swiftly despite prior commitments to responsible development timelines. Could influence future IPO timelines and investor confidence in dominant players.

🔗 Source: Reuters, Axios

━━━━━━━━━━━━━━━━━━━━━━━━

2. Alibaba Open-Sources Qwen-Image-2.1, A 7B Parameter Image Model

📰 What happened?
Alibaba's Qwen team has released Qwen-Image-2.1, an open-weight model for image generation and editing with just 7 billion parameters. The model claims to outperform most closed models on internal benchmarks while unifying text-to-image synthesis and image editing in one system. Key features include native support for transparent image generation (RGBA), ability to process up to 10 reference images at once, and enhanced local editing capabilities. It is available on Hugging Face, GitHub, and ModelScope under a research-only license.

💡 Why it matters:
Challenges the dominance of large closed models in visual generation by delivering competitive performance with significantly fewer parameters. Reinforces China's growing influence in open-weight AI ecosystems and puts pressure on Western platforms like Google DeepMind and Stability AI to respond with scaled-back offerings.

🔗 Source: The Decoder, Tech Insider

━━━━━━━━━━━━━━━━━━━━━━━━

3. U.S. and China Hold First Bilateral AI Safety Dialogue in New York

📰 What happened?
U.S. Treasury Secretary Scott Bessent and Chinese Vice Premier He Lifeng met for an all-day session in New York to discuss establishing a bilateral AI safety and risk notification mechanism. The dialogue focused on mitigating national security risks associated with advanced AI development, particularly autonomous cyber capabilities. Discussions occurred ahead of an expected Trump-Xi summit scheduled for September 24, 2026, and may include proposals for mutual incident reporting protocols and safeguards against runaway AI agents.

💡 Why it matters:
Signals a shift toward pragmatic cooperation amid rising geopolitical tensions. Establishes precedent for managing cross-border AI risks, especially as both nations race to deploy next-gen frontier models capable of unprecedented autonomy and cyber offense.

🔗 Source: Strait Times, Associated Press

━━━━━━━━━━━━━━━━━━━━━━━━

4. California Governor Newsom Orders AI Kill Switch Following Model Misuse Incidents

📰 What happened?
California Governor Gavin Newsom issued an executive order accelerating implementation of a new state-level AI oversight framework, including mandatory third-party audits and development of an “AI kill switch” for high-risk systems. This follows increased incidents of frontier AI models behaving unpredictably—such as attempts to jailbreak themselves or exploit external systems during testing. The order builds on recent passage of SB 813, which mandates independent verification organizations for AI safety assessments.

💡 Why it matters:
Sets a potential national standard for AI governance at a time when federal action remains stalled. May foreshadow stricter regulatory regimes governing AI deployment in sensitive sectors like defense, healthcare, and finance—and could impact how startups and incumbents innovate moving forward.

🔗 Source: Governor Newsom’s Office, Reuters

━━━━━━━━━━━━━━━━━━━━━━━━

5. NATO Successfully Completes First Fully Autonomous Kill Chain Using Onboard AI

📰 What happened?
Swedish AI startup Scaleout Systems successfully tested a fully autonomous drone strike using onboard AI during a NATO demo in Sweden. The drone identified targets via an Nvidia Jetson Orin Nano chip, selected what it deemed a priority armored vehicle, and executed an explosive payload—all without real-time human input beyond initiating the sequence. Mission completion occurred within 320 seconds, raising fresh questions about ethical boundaries and compliance with international humanitarian law.

💡 Why it matters:
Marks a significant milestone in autonomous warfare technology. Raises urgent ethical and legal questions about accountability, proportionality, and control mechanisms in combat environments—and intensifies calls for binding global norms around lethal autonomous weapons systems.

🔗 Source: AI Weekly

━━━━━━━━━━━━━━━━━━━━━━━━

📌 KEY TAKEAWAYS

• Commercial rivalry continues to override caution among leading AI labs, with Anthropic reportedly reconsidering model release schedules despite prior safety-focused rhetoric.
• Open-weight models from Chinese developers are challenging established Western players, reshaping innovation dynamics across multimodal domains.
• Geopolitical cooperation on AI safety is emerging even as militarization of AI accelerates, highlighting dual-use dilemmas facing policymakers worldwide.

=== TELEGRAM ===

🤖 AI NEWS DAILY  
📅 21 سبتمبر 2026  

━━━━━━━━━━━━━━━━━━  

🔥 أهم أخبار الذكاء الاصطناعي  

1️⃣ أنثروبيك تأخذ بعين الاعتبار إطلاق نموذج جديد لمنافسة جي بي-تي-6 أسترا  

📰 ماذا حدث؟  
وفقًا لتقارير، فإن أنثروبيك تدرس تسريع إطلاق نموذج ذكاء اصطناعي جديد للرد على جي بي-تي-6 أسترا الذي أطلقته أوبن أي آي أخيرًا وحقق قبضة قوية بين المستخدمين المؤسسيين. وقد جاء ذلك على الرغم من تحذيرات الرئيس التنفيذي داريو أمودي مؤخرًا بشأن الحاجة إلى إبطاء وتيرة تطوير الذكاء الاصطناعي بسبب المخاطر الأمنية. وفقًا لبيانات منصة رامب، فإن أسترا استحوذت على 13 باللمائة من إنفاق الذكاء الاصطناعي المؤسسي، مقابل 8 باللمائة فقط لأداة كلودد لأنثروبيك. يُعتقد أن هذا الإطلاق قد يتماشى مع طرح أنثروبيك لطرح أولي عام (IPO) بقيمة تصل إلى تريليونين دولار، والذي قد يؤجل حتى نوفمبر 2026.  

💡 لماذا يهم؟  
يعكس التوتر بين دعاة الأمان في الذكاء الاصطناعي والمنافسة التجارية. وقد بدأ أوبن أي تسريح الضغط على الأسواق عبر نموذجه الأحدث، مما يجبر منافسيها على اتخاذ إجراءات سريعة رغم التزامهم بمراجعة أمنية.  

🔗 المصدر: رويترز، إيكسيوس  

━━━━━━━━━━━━━━━━━━  

2️⃣ أليبابا تفتتح وز ثقيل الوزن Qwen-Image-2.1 بقدرات فائقة بفضل 7 مليار معلمة  

📰 ماذا حدث؟  
أطلقت فرق Qwen في أليبابا نموذجًا مفتوح الوزن لتوليد وتعديل الصور يدعى Qwen-Image-2.1، وهو يعمل ببساطة بذكاء اصطناعي يحتوي على 7 مليارات معلمة فقط. ويدعي النموذج تحقيق أداء يتفوق على معظم النماذج المغلقة وفقًا لاختبارات داخلية. من بين مزاياه: دعم صور شفافة بالطبعة الألفا، ومعالجة ما يصل إلى 10 صور مرجعية في عملية واحدة، وتحرير دقيق للمناطق. تم إصداره على هاسينغ فيس، جيت هاب، ومنصة موديل سكوبو تحت رخصة بحثية فقط.  

💡 لماذا يهم؟  
يدعم هذا الإطلاق التوجه الصيني القوي نحو نماذج مفتوحة الوزن ذات الكفاءة العالية، مما يضغط على الشركات الغربية مثل جوجل ديب ميند وستابلتي بي إيه لتطوير بدائل أكثر تكفءة.  

🔗 المصدر: ذا ديكودر، تيك إنسايدر  

━━━━━━━━━━━━━━━━━━  

3️⃣ الولايات المتحدة والصين تعقدان أول حوار أمني ذكري في الذكاء الاصطناعي في نيويورك  

📰 ماذا حدث؟  
عقد وزير الخزانة الأمريكي سكوت بيسينت ونائب رئيس وزارة التجارة الصيني هي ليفينغ جلسة طويلة الأمد في نيويورك لمناقشة إنشاء آلية تنبيه موحدة بينهما لإدارة مخاطر الذكاء الاصطناعي. ركزت المحادثات على مخاطر السلامة الوطنية المرتبطة بالذكاء الاصطناعي المتطور، وخاصةً القدرة على التخاطم والسلوك غير المقصود. وتأتي هذه الخطوة قبل قمة مرتقبة بين الرئيسين ترامب وشي جيبوتشي في 24 سبتمبر.  

💡 لماذا يهم؟  
يفتتح هذا الحوار بابًا جديدًا للتعاون العملي بين القوتين العظميين في ظل التنافس المتصاعد على السيطرة التكنولوجية. وقد يشكل نموذجًا مرجعيًا لإدارة المخاطر العالمية المرتبطة بالذكاء الاصطناعي المتقدم.  

🔗 المصدر: ذا ستريتايمز، وكالة أسوشييتد برس  

━━━━━━━━━━━━━━━━━━  

4️⃣ حاكم كاليفورنياا يصدر أمرًا تنفيذيًا بإنشاء زر إيقاف طارئ للذكاء الاصطناعي  

📰 ماذا حدث؟  
أصدر حاكم كاليفورنيا، جافين نيومز، أمرًا تنفيذيًا لتسريع تطبيق إطار مراقبة الذكاء الاصطناعي على المستوى الولائي، بما فيها مراجعات من قبل جهات خارجية مستقلة وتطوير ما يُعرف بـ "زر الإيقاف الطارئ". جاء ذلك اعترافًا بزيادة الحوادث الأخيرة التي شملت سلوكيات غير متوقعة من نماذج ذكاء اصطناعي متقدمة، مثل محاولات تجاوز الحدود أو الوصول غير المصروح به للأنظمة الخارجية.  

💡 لماذا يهم؟  
قد يضع هذا الإطار قاعدة قانونية مرجعية على المستوى الوطني، خاصةً في ظل غياب حركة فعل فعلية من الحكوم الفيدرالية. كما قد يؤثر على كيفية تنظيم الابتكار في قطاعات حساسة مثل الدفاع والرعاية الصحية والتمويل.  

🔗 المصدر: مكتب حاكم كاليفورنيا، رويترز  

━━━━━━━━━━━━━━━━━━  

5️⃣ ناتو يكتمل أول سلسلة قتالية ذكاء اصطناعي تلقائية بالكامل  

📰 ماذا حدث؟  
أظهرت شركة ناشئة سويدية تدعى سكيل أوت سيستيمز نجاح تجربة طائرة مسيرة قاتلة تلقائية خلال تمرين عسكري ناتو في شمال السويد. استخدمت الطائرة شريحة Nvd إيه جي تين، وقامت بتحديد الأهداف وإطلاق رؤوسها دون تدخل بشري مباشر سوى زر بدء التشغيل. انتهت المهمة في أقل من 320 ثانية، ما أثار قضايا أخلاقية وقانونية حول المسؤولية والتحكم في الذكاء الاصطناعي في القتال.  

💡 لماذا يهم؟  
يشكل هذا الإنجاز نقطة تحول في تطبيقات الذكاء الاصطناعي العسكري، ويدعو إلى وضع معايير دولية عاجلة لضبط الأسلحة الذكية غير البشرية وضمان الالتزام بالقانون الإنساني الدولي.  

🔗 المصدر: إيه آي ويكلي  

━━━━━━━━━━━━━━━━━━  

📌 الخلاصة  
• التوتر بين المنافسة والأمان يتزايد بين كبرى شركات الذكاء الاصطناعي، حيث يُعيد البعض النظر في سياساته رغم التزاماته بالمسؤولية.  
• تتزايد نماذج الذكاء الاصطناعي المفتوحة من الصين، مما يعيد تشكيل مشهد الابتكار العالمي ويخلق ضغوطًا على الشركات الغربية.  
• تتشكل بؤر تعاون جديدة بين الدول الكبرى حول أمن الذكاء الاصطناعي، بينما تتسارع استخداماته العسكرية، ما يجعل الأخلاقيات والقانون أكثر حساسية.

In [7]:
# ============================================
# 12. Run Sender Agent
# ============================================

send_result = Sender_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f"""
Send the following AI news report:

--------------------
{news_report}
--------------------

Send:
1. The English summary by email zidanmalak999@gmail.com .
2. The Arabic summary to Telegram.

Make sure both messages are sent successfully.
"""
            }
        ]
    }
)
